In [1]:
import spacy
from FlagEmbedding import FlagModel
from elasticsearch import Elasticsearch
from elasticsearch import helpers

from tqdm import tqdm

In [2]:
# IndexText uses an embedding model to generate vector embeddings of an input text and stores these embeddings in a 
# vector database using Elasticsearch. The input text in split into chunks of multiple sentences, then embedded and indexed.
# The vector database takes the form (text_chunk_i, vector_embedding_i) for the ith entry of the index.

In [4]:
# helper functions
def remove_newline(text):
# removes newline characters, "\n", from text
# text: list of paragraphs in the text
    
    for i in range(len(text)):
        text[i] = " ".join(text[i].split())

    return text

def embed_index_text(text_chunks, client):
    # chunks: list of m elements that contain n sentences each
    # client: instance of Elasticsearch client used to create the index

    # embed chunks
    chunk_embeddings = model.encode(text_chunks).tolist()

    # define the format of the data to be indexed as pairs (chunk of text, chunk embeddings)
    docs = [
        {
            '_op_type': 'index',
            '_index': 'les_miserables_index',
            '_source': {
                "chunk" : t, 
                "embedding_vector" : v
            }
        } for t, v in zip(text_chunks, chunk_embeddings)
    ]
    
    # index in bulk
    res = helpers.bulk(client, docs)
    # print(res)

def createIndex(client, index_name):
    # client: instance of Elasticsearch client
    # index_name (str): index name

    # ensure that there is no previously defined index under the index_name
    if (client.indices.exists(index = index_name)):
        client.indices.delete(index = index_name)

    # define the format of index: chunk of text and embedding vectors
    # custom mapping that defines the expected types of indices features
    # define mapping parameters for the "chunk" and "embedding_vector" fields
    # define "vector_dim"
    mappings = {
        "properties": {
            "chunk": {
                "type": "text"
            }, 
            "embedding_vector": {
                "index": True, 
                "type": "dense_vector", 
                "dims": 512, 
                "similarity": "cosine",
            }
        }
    }
        
    # create index
    client.indices.create(index = index_name, mappings = mappings)


In [28]:
class IndexText:
# class methods: 
    # __preProcessInput: reads a text from a file_path, and splits the text into paragraphs eliminating newline characters
    # chunkEmbedIndex: splits the pre-processed text into chunks and indexes the chunks using Elasticsearch
    
    def __init__(self, client):
        # instance variables defined below
        
        # elastic search client
        self.client = client

    def __preProcessInput(self, file_path):
        # text: text file

        with(open(file_path, "r")) as text_file:
            text = text_file.read()

        # split text in paragraphs
        text = text.split("\n\n")

        # using the helper function "remove_newline" to eliminate "\n" characters from the text
        text = remove_newline(text)

        return text

    def chunkEmbedIndex(self, file_path, sentence_limit, chunk_limit, min_characters):
        text = self.__preProcessInput(file_path)
        
        # Load pretrained English Language Model to separate the text into sentences
        nlp = spacy.load('en_core_web_sm') 

        chunks = []
        sentences = []

        # generate doc pipeline with nlp
        # allows to process the data as a stream and buffer the paragraphs in batches instead of one by one
        doc_pipeline = nlp.pipe(text, batch_size = 5, n_process = 1)

        # split the doc into sentences and create chunks that contain at least n = "sentence_limit" sentences
        # once m = "chunk_limit" chunks have been collected, create chunk embeddings and index them
        for doc in tqdm(doc_pipeline):
            for sent in doc.sents:
                sentences.append(sent.text)

                if len(sentences) >= sentence_limit:
                    chunk = " ".join(sentences)

                    # if the number of characters in the current chunk is less than the minimum required, 
                    # then add another sentence and count again before embedding the text
                    if len(chunk) < min_characters:
                        continue

                    # once the chunk has the minimum length, append it to chunks
                    chunks.append(chunk)
                    # remove the first sentence and keep the other two to overlap with the following sentence
                    sentences = sentences[1:]

                if len(chunks) == chunk_limit:
                    # embed and index
                    embed_index_text(chunks, self.client)

                    # clear list of chunks
                    chunks = []

        # if there are sentences/chunks that haven't been embedded and indexed, do so
        if len(sentences) != 0:
            # append sentences to remaining chunks
            chunks.append(" ".join(sentences))

            embed_index_text(chunks, self.client)

            sentences = []
            chunks = []


class IndexQuery:

    def __init__(self, client):
        # Python client for Elasticsearch
        self.client = client

    def __embedQueries(self, queries):
        # queries: list of text queries
        # returns a list of vector embeddings for each query 
        q_embeddings = model.encode_queries(queries)

        return q_embeddings.tolist()[0]

    def knnSearch(self, index_name, queries):
        # index_name: str
        # queries: list of text queries
        
        query_vector = self.__embedQueries(queries)
        score_chunk = []
        
        resp_knn = self.client.search(
            index = index_name,
            # size = 3,  # number of top global results after combining shard results
            query = {
                "knn": {
                    "field": "embedding_vector",
                    "query_vector": query_vector,
                    "k": 10,  # nearest neighbours to return from each shard
                    "num_candidates": 100,  # number of nearest neighbor candidates to consider per shard while doing knn search
                }
            },
        )

        # return scores and chunks as tuples in a list
        
        for hit in resp_knn["hits"]["hits"]:
            score_chunk.append((hit["_score"], hit["_source"]["chunk"]))
            # print(hit["_score"], hit["_source"]["chunk"])
            # print()

        return score_chunk

        
        


In [6]:
# embedding model
# load bge language model to embed the text in chunks
model = FlagModel('BAAI/bge-small-zh-v1.5', use_fp16 = True)

tokenizer_config.json:   0%|          | 0.00/367 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/110k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/439k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/95.8M [00:00<?, ?B/s]

In [8]:
# instantiate Python client for Elasticsearch
client = Elasticsearch("http://elasticsearch:9200")

In [9]:
client.indices.exists(index = "les_miserables_index")

HeadApiResponse(True)

In [7]:
# create index
# createIndex(client, "les_miserables_index")

In [11]:
# instantiate IndexText class
# les_miserables = IndexText(client)

In [28]:
# text = les_miserables.preProcessInput("../data/LesMiserables.txt")

In [12]:
# define chunk parameters before calling the chunkEmbedIndex method in the IndexText class

# minimum number of sentences in a chunk
max_sentence = 6

# number of text chunks to embed and index
max_chunk = 64

# minimum number of characters in a chunk
min_characters = 128

In [13]:
# index the text
# les_miserables.chunkEmbedIndex("../data/LesMiserables.txt", max_sentence, max_chunk, min_characters)

31it [00:00, 70.11it/s]You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
13942it [3:10:11,  1.22it/s]


In [30]:
query_class = IndexQuery(client)

In [31]:
knn_res = query_class.knnSearch("les_miserables_index", ["how old is the bishop when he dies?"])

In [41]:
knn_res

[(0.8369942,
  'he asked. “Certainly, sir; you see, the prefecture of to-day was the bishop’s palace before the Revolution. M. de Conzié, who was bishop in ’82, built a grand hall there. It is in this grand hall that the court is held.” On the way, the bourgeois said to him:— “If Monsieur desires to witness a case, it is rather late. The sittings generally close at six o’clock.” When they arrived on the grand square, however, the man pointed out to him four long windows all lighted up, in the front of a vast and gloomy building. “Upon my word, sir, you are in luck; you have arrived in season.'),
 (0.834157,
  'Did I exist before my birth? No. Shall I exist after death? No. What am I? A little dust collected in an organism. What am I to do on this earth?'),
 (0.8313389,
  '“Is it there that the Assizes are held?” he asked. “Certainly, sir; you see, the prefecture of to-day was the bishop’s palace before the Revolution. M. de Conzié, who was bishop in ’82, built a grand hall there. It is

In [14]:
resp = client.search(
    index = "les_miserables_index",
    query = {
        "bool": {
            "must": [
                {
                    "match": {
                        "chunk": "gavroche",
                    }
                },
                {
                    "match": {
                        "chunk": "javert",
                    }
                }
            ]
        }
    },
    source = ["chunk"]
)

In [17]:
resp.keys()

dict_keys(['took', 'timed_out', '_shards', 'hits'])

In [19]:
resp["hits"]["hits"]

[{'_index': 'les_miserables_index',
  '_id': 'lybw-5MBJCf2OsYvXFUQ',
  '_score': 9.452269,
  '_source': {'chunk': 'Javert, with his back to the post, and so surrounded with ropes that he could not make a movement, raised his head with the intrepid serenity of the man who has never lied. “He is a police spy,” said Enjolras. And turning to Javert: “You will be shot ten minutes before the barricade is taken.” Javert replied in his most imperious tone:— “Why not at once?” “We are saving our powder.” “Then finish the business with a blow from a knife.” “Spy,” said the handsome Enjolras, “we are judges and not assassins.” Then he called Gavroche:— “Here you! go about your business! Do what I told you!” “I’m going!” cried Gavroche.'}},
 {'_index': 'les_miserables_index',
  '_id': 'mCbw-5MBJCf2OsYvXFUQ',
  '_score': 9.247811,
  '_source': {'chunk': '“He is a police spy,” said Enjolras. And turning to Javert: “You will be shot ten minutes before the barricade is taken.” Javert replied in his mo